In [0]:
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.getOrCreate()

# df = spark.range(5)

# display(df)

In [0]:
# import sys
# print(sys.executable)

# 식봄(foodspring.co.kr) 키워드 검색 크롤러

키워드로 검색해서 **상품명 · 가격 · 단위(중량/용량)** 를 수집합니다.

- 검색 URL: `https://www.foodspring.co.kr/search/all?key={키워드}`
- 사이트가 Next.js + GraphQL SPA라서 **Selenium**으로 렌더링 후 DOM을 파싱합니다.

## 1. 패키지 설치 (최초 1회만)

In [0]:
# !pip install selenium pandas

## 2. 라이브러리 import

In [0]:
import time
import re
import urllib.parse

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## 3. 헬퍼: Selenium 드라이버 & 스크롤

In [0]:
def build_driver(headless: bool = True) -> webdriver.Chrome:
    """Selenium Chrome 드라이버 생성."""
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,2000")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    drv = webdriver.Chrome(options=opts)
    drv.set_page_load_timeout(20)       # 페이지 로드 20초 안되면 예외
    drv.set_script_timeout(20)
    return drv


def scroll_to_bottom(driver, max_rounds: int = 12, pause: float = 1.2):
    """무한스크롤: 더 이상 높이가 늘지 않을 때까지 내림."""
    last = 0
    for _ in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)
        h = driver.execute_script("return document.body.scrollHeight")
        if h == last:
            break
        last = h

## 4. 카드 텍스트 파서 (가격 · 단위 추출)

In [0]:
PRICE_RE = re.compile(r"([0-9][0-9,]*)\s*원")

UNIT_TOKENS = r"kg|g|ml|l|L|개입|개|봉|팩|병|ea|EA|Box|box|포|장|매|구|미|호|박스"
UNIT_RE = re.compile(rf"(\d+(?:\.\d+)?)\s*({UNIT_TOKENS})", re.IGNORECASE)
STD_UNIT_CHARS = {"g", "kg", "ml", "l"}

_UNIT_ONLY_RE = re.compile(
    rf"^[\d\s/·xX×.,]*\s*({UNIT_TOKENS})(\s*/\s*(?:EA|ea|Box|box))?\s*$",
    re.IGNORECASE,
)
_NUM_ONLY_RE = re.compile(r"^[\d\s/·xX×.,]+$")
_BADGE_LINES = {"쿠폰가", "정가", "할인가", "판매가", "BEST", "NEW", "품절", "단독"}

# 라인 끝의 배너 단어 → 카테고리 배너로 간주하고 상품명 후보에서 제외
# "계란·알 BEST", "참기름 BEST", "수입산 돼지고기 4위", "주간 BEST" 등
_BANNER_TAIL_RE = re.compile(
    r"(\s(BEST|NEW|HOT|단독|추천|이벤트|기획전)\s*$"
    r"|\d+\s*위\s*$"
    r"|\s주간\s*BEST\s*$)",
    re.IGNORECASE,
)

_TOKEN_SPLIT_RE = re.compile(r"[\s·/\-_(),\[\]+,]+")


def _is_meta_line(ln: str) -> bool:
    if not ln:
        return True
    if "원" in ln or "%" in ln:
        return True
    if ln in _BADGE_LINES:
        return True
    if _UNIT_ONLY_RE.fullmatch(ln):
        return True
    if _NUM_ONLY_RE.fullmatch(ln):
        return True
    if _BANNER_TAIL_RE.search(ln):
        return True
    return False


EXCLUDE_KEYWORDS = [
    "떡볶이", "만두", "라면", "피자", "김밥", "햄버거", "도시락", "샐러드",
    "죽", "찌개", "탕", "볶음", "조림", "구이", "전골", "전",
    "스낵", "과자", "쿠키", "케이크", "빵", "샌드위치", "버거",
    "소스", "드레싱", "양념", "분말스프", "스프", "수프",
    "음료", "주스", "차", "커피", "라떼",
    "맛", "향",
    "비스킷", "와플", "프레첼",
]

SUFFIX_BAD = [
    "맛", "향", "과자", "스낵", "핫도그", "튀김", "전",
    "버거", "피자", "샌드", "캔디", "쿠키",
    "젓",
    "링",
    "롤", "까스",   # 치즈롤까스·생선까스 등 가공조리식품
]

BUNDLE_RE = re.compile(
    r"(\d+\s*\+\s*\d+|세트|묶음|박스|BOX|기획|선물|대용량|벌크|증정|패키지)",
    re.IGNORECASE,
)
BAD_GRADE_RE = re.compile(
    r"(임박|B\s*급|못난이|자투리|리퍼|반품|랜덤|흠집|파지|훼손|이월)",
    re.IGNORECASE,
)

RAW_LV1 = {"채소류", "육류·계란", "어패류", "과일·견과", "곡류·면"}
PROCESSED_HINTS = [
    "양념", "절임", "장아찌", "통조림", "캔", "훈제",
    "파우더", "분말", "농축", "엑기스", "시럽", "페이스트",
    "조미", "소금간", "간장", "젓갈",
    "다시다", "다시팩", "다시", "스톡", "육수", "맛가루", "맛소금",
    "다짐", "다진", "민찌", "민치", "슬라이스", "잘게썬",
]

TRAP_PREFIXES = {
    "갈릭", "허브", "칠리", "데리야끼", "까르보", "트러플",
    "스파이시", "스모크", "허니", "데미", "토마토",
    "마늘맛", "양념"
}

LV2_PREFER = {
    "가루양념": ["가루", "분말", "파우더"],
    "당류":     ["설탕", "당"],
    "기름":     ["기름", "오일"],
    "장류":     ["장"],
    "식초·발효": ["식초"],
    "액젓·국물조미료": ["액젓", "다시팩"],
    "버터":     ["버터"],
    "치즈":     ["치즈"],
    "두부·묵":  ["두부", "묵"],
    "밀가루·전분": ["밀가루", "전분"],
    "면류":     ["면"],
    "돼지고기": ["삼겹", "목살", "앞다리", "뒷다리", "갈비", "안심", "등심"],
    "닭고기":   ["닭", "치킨"],
    "소고기":   ["등심", "안심", "갈비", "양지", "차돌"],
}

SYNONYMS = {
    "소고기": ["쇠고기", "한우", "수입소", "호주산", "미국산", "찹스테이크"],
    "쇠고기": ["소고기", "한우"],
    "닭고기": ["닭", "치킨"],
    "돼지고기": ["돈육", "삼겹살용"],
    "계란": ["달걀"],
    "감자": ["감자(국산)"],
}

INGREDIENT_NAMES: set = set()


def _get_name_variants(재료명: str, keyword: str) -> list:
    variants = {재료명, keyword}
    variants.update(SYNONYMS.get(재료명, []))
    variants.update(SYNONYMS.get(keyword, []))
    return [v for v in variants if v]


def has_modifier_prefix(name: str, keyword: str) -> str:
    idx = name.find(keyword)
    if idx <= 0:
        return ""
    before = name[:idx]
    tokens = [t for t in _TOKEN_SPLIT_RE.split(before) if t]
    if not tokens:
        return ""
    last = tokens[-1]
    if last == keyword:
        return ""
    if last in INGREDIENT_NAMES or last in TRAP_PREFIXES:
        return last
    for L in (4, 3, 2):
        if len(last) > L:
            tail = last[-L:]
            if tail == keyword:
                continue
            if tail in INGREDIENT_NAMES or tail in TRAP_PREFIXES:
                return tail
    return ""


def is_contaminated(product_name: str, keyword: str, lv1: str = "") -> tuple:
    if keyword not in EXCLUDE_KEYWORDS:
        for w in EXCLUDE_KEYWORDS:
            if w in keyword:
                continue
            if w in product_name:
                return True, f"가공품키워드:{w}"

    pref = has_modifier_prefix(product_name, keyword)
    if pref:
        return True, f"앞수식어:{pref}"

    idx = product_name.find(keyword)
    if idx >= 0:
        tail = product_name[idx + len(keyword): idx + len(keyword) + 4]
        for suf in SUFFIX_BAD:
            if suf in keyword:
                continue
            if tail.startswith(suf):
                return True, f"접미어:{keyword}{suf}"

    if BUNDLE_RE.search(product_name):
        return True, "세트/묶음"
    if BAD_GRADE_RE.search(product_name):
        return True, "B급/임박/리퍼"

    if lv1 in RAW_LV1:
        for w in PROCESSED_HINTS:
            if w in keyword:
                continue
            if w in product_name:
                return True, f"원물카테고리에가공:{w}"
    return False, ""


def score_product(name: str, 재료명: str, keyword: str,
                  lv1: str, lv2: str,
                  unit_num, unit_chr: str, price) -> float:
    score = 0.0
    if name.startswith(재료명) or name.startswith(keyword):
        score += 2
    elif (재료명 in name[:8]) or (keyword in name[:8]):
        score += 1
    if unit_chr and unit_chr.lower() in STD_UNIT_CHARS:
        score += 1
    try:
        if unit_num and 0 < float(unit_num) <= 10000:
            score += 0.5
    except (TypeError, ValueError):
        pass
    for sep in ["·", "/", " "]:
        for tok in [t.strip() for t in str(lv2).split(sep) if t.strip()]:
            if len(tok) >= 2 and tok in name:
                score += 1
                break
    for tok in LV2_PREFER.get(str(lv2), []):
        if tok in name:
            score += 2
            break
    score -= max(0.0, (len(name) - 20) / 20)
    if price and price > 100_000:
        score -= 1
    return round(score, 3)


def parse_card(text: str, keyword: str = "") -> dict:
    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    prices = [int(m.replace(",", "")) for m in PRICE_RE.findall(text)]
    unit_match = UNIT_RE.search(text)

    cand = [ln for ln in lines if not _is_meta_line(ln) and len(ln) >= 2]

    name = ""
    if keyword:
        kw_lines = [ln for ln in cand if keyword in ln]
        if kw_lines:
            name = max(kw_lines, key=len)
    if not name and cand:
        body = cand[1:] if len(cand) > 1 else cand
        name = max(body, key=len)

    price = min(prices) if prices else None
    list_price = max(prices) if len(prices) > 1 else None

    if unit_match:
        unit_num_raw = unit_match.group(1)
        unit_chr = unit_match.group(2)
        unit_full = f"{unit_num_raw}{unit_chr}"
        try:
            unit_num = float(unit_num_raw)
            if unit_num.is_integer():
                unit_num = int(unit_num)
        except ValueError:
            unit_num = None
    else:
        unit_full, unit_num, unit_chr = "", None, ""

    return {
        "상품명": name,
        "가격": price,
        "정가": list_price,
        "단위": unit_full,
        "단위_수치": unit_num,
        "단위_문자": unit_chr,
        "raw": " | ".join(lines[:6]),
    }

## 5. 메인 크롤링 함수

In [0]:
from datetime import date
from collections import Counter
from selenium.common.exceptions import StaleElementReferenceException

SOURCE_NAME = "식봄"

def _fuzzy_match(variant: str, product_name: str) -> bool:
    """
    퍼지 키워드 매칭:
    1) 정확 부분문자열 매칭 (v in name)
    2) 공백 제거 후 매칭 ("둥지물냉면" in "둥지냉면물냉면" → "둥지물냉면" in "둥지냉면물냉면")
    3) 키워드의 모든 글자가 상품명에 순서대로 등장 (subsequence)
    """
    # 1) 정확 매칭
    if variant in product_name:
        return True
    
    # 2) 공백 제거 후 매칭
    name_nospace = product_name.replace(" ", "")
    if variant in name_nospace:
        return True
    
    # 3) 키워드 글자가 상품명에 순서대로 모두 등장 (subsequence match)
    #    예: "둥지물냉면" → 둥,지,물,냉,면 이 순서대로 name_nospace에 있는지
    if len(variant) >= 3:  # 너무 짧으면 오탐 위험
        it = iter(name_nospace)
        if all(ch in it for ch in variant):
            return True
    
    return False

def crawl_foodspring(
    재료명: str,
    keyword: str = None,
    headless: bool = True,
    max_scroll: int = 4,
    meta: dict = None,
    driver=None,
    debug: bool = False,
):
    keyword = keyword or 재료명
    meta = meta or {}
    lv1 = str(meta.get("lv1", "") or "")
    lv2 = str(meta.get("lv2", "") or "")
    variants = _get_name_variants(재료명, keyword)

    url = f"https://www.foodspring.co.kr/search/all?key={urllib.parse.quote(keyword)}"
    owns_driver = driver is None
    if owns_driver:
        driver = build_driver(headless=headless)

    drop = Counter()
    drop_samples = {}

    def _note(stage, name):
        drop[stage] += 1
        drop_samples.setdefault(stage, []).append(name)

    try:
        driver.get(url)
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/goods/detail/']"))
            )
        except Exception:
            return (pd.DataFrame(), {"reason": "no-results"}) if debug else pd.DataFrame()

        scroll_to_bottom(driver, max_rounds=max_scroll, pause=0.8)
        time.sleep(0.3)

        anchors = driver.find_elements(By.CSS_SELECTOR, "a[href*='/goods/detail/']")
        seen, rows = set(), []
        today = date.today().isoformat()
        total_cards = 0

        for a in anchors:
            try:
                href = a.get_attribute("href") or ""
            except StaleElementReferenceException:
                _note("stale", "")
                continue
            m = re.search(r"/goods/detail/(\d+)", href)
            if not m:
                continue
            gid = m.group(1)
            if gid in seen:
                continue
            seen.add(gid)
            total_cards += 1

            try:
                try:
                    card = a.find_element(By.XPATH, "./ancestor::li[1]")
                except Exception:
                    card = a
                txt = card.text or a.text
            except StaleElementReferenceException:
                _note("stale", "")
                continue

            if not txt.strip():
                continue

            # parse_card: 동의어 중 가장 흔한 표기로 시도
            parsed = parse_card(txt, keyword=keyword)
            name = parsed["상품명"]
            if not name:
                _note("상품명추출실패", txt[:30])
                continue

            # 퍼지 키워드 매칭: 공백 제거 + subsequence 매칭 추가
            matched_variant = next((v for v in variants if _fuzzy_match(v, name)), "")
            if not matched_variant:
                _note("키워드미포함", name)
                continue

            # 오염 검사는 매칭된 표기로 수행 (앞-수식어 검사 정확도 향상)
            bad, reason = is_contaminated(name, matched_variant, lv1=lv1)
            if bad:
                _note(f"오염({reason})", name)
                continue

            if not parsed["단위_문자"]:
                _note("단위미파싱", name)
                continue
            try:
                if not (parsed["단위_수치"] and float(parsed["단위_수치"]) > 0):
                    _note("단위수치이상", name)
                    continue
            except (TypeError, ValueError):
                _note("단위수치이상", name)
                continue
            if parsed["가격"] is None:
                _note("가격없음", name)
                continue

            score = score_product(
                name, 재료명, matched_variant, lv1, lv2,
                parsed["단위_수치"], parsed["단위_문자"], parsed["가격"],
            )
            rows.append({
                "ing_id": meta.get("ing_id", ""),
                "재료명": 재료명,
                "lv1": lv1,
                "lv2": lv2,
                "kca_category_ref": meta.get("kca_category_ref", ""),
                "상품명": name,
                "가격": parsed["가격"],
                "단위": parsed["단위"],
                "단위_수치": parsed["단위_수치"],
                "단위_문자": parsed["단위_문자"],
                "score": score,
                "출처": SOURCE_NAME,
                "수집일": today,
                "goods_id": gid,
                "url": href.split("?")[0],
            })

        df = pd.DataFrame(rows, columns=[
            "ing_id", "재료명", "lv1", "lv2", "kca_category_ref",
            "상품명", "가격", "단위", "단위_수치", "단위_문자",
            "score", "출처", "수집일", "goods_id", "url",
        ])
        if debug:
            return df, {
                "총카드수": total_cards,
                "통과수": len(df),
                "탈락": dict(drop),
                "샘플": {k: v[:3] for k, v in drop_samples.items()},
            }
        return df
    finally:
        if owns_driver:
            driver.quit()

## 6. 실행 — 크롤링 + 후처리 + 1:1 매칭

`재료명_목록500.csv`에서 `OFFSET ~ OFFSET+LIMIT` 범위 재료를 크롤링하고,
가격 IQR · goods_id 중복 처리 · 재료당 1개 최적 상품 매칭까지 일괄 수행.

- 결과는 메모리(`candidates`, `clean`, `best`)에만 보관 (CSV 저장 없음)
- 다음 배치는 `OFFSET`을 20·40·60… 으로 바꿔 재실행
- 매핑 결과는 `ing_id` 오름차순으로 표시
- 디버그: `DEBUG_TARGETS` 에 재료명 넣으면 단계별 탈락 사유 출력

In [0]:
import time as _t

# ▼ 옵션 ────────────────────────────────────────────────────
CSV_PATH = "완1_cleaned_2차.csv"
OFFSET = 0
LIMIT = None
DRIVER_RESET_EVERY = 30
DEBUG_ALL = False
DEBUG_TARGETS = set()
# ──────────────────────────────────────────────────────────


def load_ingredients(path: str) -> pd.DataFrame:
    """재료 CSV 로드 + INGREDIENT_NAMES 사전 등록."""
    df = (
        pd.read_csv(path, encoding="utf-8")
          .dropna(subset=["std_name"])
          .reset_index(drop=True)
    )
    INGREDIENT_NAMES.clear()
    INGREDIENT_NAMES.update(df["std_name"].astype(str).str.strip().tolist())
    return df


def crawl_batch(ing_df: pd.DataFrame, offset: int, limit: int,
                debug_targets: set, debug_all: bool = False) -> pd.DataFrame:
    """OFFSET~OFFSET+LIMIT 배치 크롤링. 드라이버 재사용 + 재시도 + 주기적 재시작."""
    start, end = offset, offset + limit if limit else len(ing_df)
    batch = ing_df.iloc[start:end].copy()
    # print(f"배치: {start} ~ {end-1} (총 {len(batch)}개) / 전체 {len(ing_df)}개")
    # print(batch[["ing_id", "std_name", "lv1", "lv2"]].to_string(index=False))

    frames = []
    driver = build_driver(headless=True)
    processed = 0
    try:
        for i, row in batch.iterrows():
            ing_id = str(row.get("ing_id", "")).strip()
            재료명 = str(row["std_name"]).strip()
            if not 재료명:
                continue
            meta = {
                "ing_id": ing_id,
                "lv1": row.get("lv1", ""),
                "lv2": row.get("lv2", ""),
                "kca_category_ref": row.get("kca_category_ref", ""),
            }
            use_debug = debug_all or (재료명 in debug_targets)
            print(f"[{i+1}/{len(ing_df)}] {ing_id} {재료명} (lv2={meta['lv2']})"
                  + ("  [DEBUG]" if use_debug else ""))

            df = pd.DataFrame()
            summary = None
            for attempt in (1, 2):
                try:
                    result = crawl_foodspring(
                        재료명, keyword=재료명, meta=meta, driver=driver, debug=use_debug,
                    )
                    df, summary = result if use_debug else (result, None)
                    break
                except Exception as e:
                    print(f"   !! 시도{attempt} 실패: {type(e).__name__}: {e}")
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = build_driver(headless=True)

            print(f"   -> 후보 {len(df)}건")
            if use_debug and summary:
                print(f"   [debug] 총카드={summary.get('총카드수')}, 탈락={summary.get('탈락')}")
                for stage, samples in summary.get("샘플", {}).items():
                    print(f"     - {stage}: {samples}")

            if len(df):
                frames.append(df)
            processed += 1

            if processed % DRIVER_RESET_EVERY == 0:
                try:
                    driver.quit()
                except Exception:
                    pass
                driver = build_driver(headless=True)
                print("   ... 드라이버 재시작")
            _t.sleep(0.3)
    finally:
        try:
            driver.quit()
        except Exception:
            pass

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def postprocess(candidates: pd.DataFrame) -> tuple:
    """가격 IQR + goods_id 중복 + 재료당 1:1 매칭. (clean, best) 반환."""
    if candidates is None or len(candidates) == 0:
        return pd.DataFrame(), pd.DataFrame()

    df = candidates.copy()
    df["가격"] = pd.to_numeric(df["가격"], errors="coerce")
    df["score"] = pd.to_numeric(df["score"], errors="coerce").fillna(0)
    df = df.dropna(subset=["가격"]).reset_index(drop=True)
    before = len(df)

    def _iqr_mask(s: pd.Series) -> pd.Series:
        if len(s) < 4:
            return pd.Series(True, index=s.index)
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
        return s.between(lo, hi)

    if before:
        mask = df.groupby("재료명")["가격"].transform(_iqr_mask).astype(bool)
        df = df[mask].reset_index(drop=True)
    print(f"가격 IQR 필터: {before} → {len(df)}행")

    df = (
        df.sort_values("score", ascending=False)
          .drop_duplicates("goods_id", keep="first")
          .reset_index(drop=True)
    )
    print(f"goods_id 중복 제거: {len(df)}행")

    if len(df) == 0:
        return df, pd.DataFrame()

    best = (
        df.sort_values(["재료명", "score", "가격"], ascending=[True, False, True])
          .groupby("재료명", as_index=False)
          .first()
    )
    return df, best


# === 실행 ===========================================================
ing_df = load_ingredients(CSV_PATH)
print(f"재료 {len(ing_df)}건 / INGREDIENT_NAMES 사전 {len(INGREDIENT_NAMES)}개\n")

candidates = crawl_batch(ing_df, OFFSET, LIMIT, DEBUG_TARGETS, DEBUG_ALL)
print(f"\n[크롤링 완료] 후보 {len(candidates)}행")

clean, best = postprocess(candidates)
print(f"[1:1 매칭 완료] {len(best)}개 재료")

# ing_id 오름차순 정렬 후 표시
if len(best):
    best_sorted = best.sort_values("ing_id").reset_index(drop=True)
    display(best_sorted[["ing_id", "재료명", "lv1", "lv2", "상품명", "가격", "단위", "score"]])

In [0]:
# with pd.option_context("display.max_rows", None,
#                        "display.max_colwidth", None,
#                        "display.width", None):
#     display(best_sorted[["ing_id", "재료명", "lv1", "lv2", "상품명", "가격", "단위", "score"]])

In [0]:
# 매칭/미매칭 재료 분리 → 로컬 CSV 저장
matched_ids = set(best["ing_id"].astype(str))

# matched: 재료 메타 + 매칭된 상품 정보 LEFT JOIN
matched = (
    ing_df[ing_df["ing_id"].astype(str).isin(matched_ids)]
      .merge(
          best[["ing_id", "상품명", "가격", "단위", "단위_수치", "단위_문자", "score", "url"]],
          on="ing_id", how="left",
      )
      .sort_values("ing_id")
      .reset_index(drop=True)
)

# unmatched: best에 없는 재료들 (재료 메타만)
unmatched = (
    ing_df[~ing_df["ing_id"].astype(str).isin(matched_ids)]
      .sort_values("ing_id")
      .reset_index(drop=True)
)

print(f"matched: {len(matched)}건, unmatched: {len(unmatched)}건")

matched.to_csv("완1)matched_2.csv", index=False, encoding="utf-8-sig")
unmatched.to_csv("완1)unmatched_2.csv", index=False, encoding="utf-8-sig")
print("저장 완료: matched.csv, unmatched.csv")

In [0]:
# # 테스트용 ---------------------------------------------------
# # 테스트용 ---------------------------------------------------
# # 테스트용 ---------------------------------------------------

# from selenium import webdriver
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.common.by import By
# import time

# opts = Options()
# opts.add_argument("--headless=new")
# opts.add_argument("--no-sandbox")
# opts.add_argument("--disable-dev-shm-usage")

# driver = webdriver.Chrome(options=opts)
# driver.set_page_load_timeout(30)

# t0 = time.time()
# driver.get("https://www.foodspring.co.kr/search/all?key=마늘")
# print(f"로딩 시간: {time.time()-t0:.1f}초")
# print("타이틀:", driver.title)

# time.sleep(3)
# anchors = driver.find_elements(By.CSS_SELECTOR, "a[href*='/goods/detail/']")
# print(f"상품 링크 개수: {len(anchors)}")
# if anchors:
#     print("첫 상품 텍스트:", anchors[0].text[:200])

# driver.quit()


In [0]:
# # Databricks Workspace 업로드 — 로컬 Python에서 SDK로 푸시
# # 최초 1회 준비:
# #   1) pip install databricks-sdk
# #   2) databricks configure --token   (또는 DATABRICKS_HOST / DATABRICKS_TOKEN 환경변수)

# from databricks.sdk import WorkspaceClient
# from databricks.sdk.service.workspace import ImportFormat
# import io

# w = WorkspaceClient()
# base = "/Workspace/Users/rimmyeb@gmail.com/asac_10_dataanalysis/files"

# uploads = {
#     "foodspring_best.csv":       best_sorted,   # 1:1 매칭 (재료당 1개)
#     "foodspring_clean.csv":      clean,         # IQR + dedup 정제본
#     "foodspring_candidates.csv": candidates,    # 필터 통과한 전체 후보
# }

# for name, df in uploads.items():
#     buf = io.BytesIO()
#     df.to_csv(buf, index=False, encoding="utf-8-sig")
#     buf.seek(0)
#     w.workspace.upload(
#         f"{base}/{name}",
#         buf,
#         overwrite=True,
#         format=ImportFormat.AUTO,
#     )
#     print(f"uploaded: {base}/{name}  ({len(df)} rows)")